# Mobility Backbone Definition – Version 1

## Objective
Define the interurban road backbone that will serve as the spatial decision space for the EV charging optimization model.

This layer is intended to represent the roads on which long-distance and interurban EV travel is most relevant, in line with the datathon brief. It is **not yet the final candidate siting layer** for new charging stations.

## Data source
Official Ministry road segments shapefile:
`250311 catalogoRCE2024.shp`

## Variables inspected
- `clase`: road class
- `carretera`: road identifier
- `geometry`: road segment geometry
- `pk_inicio`, `pk_fin`, `longitud`: segment structure

## Road classes found
- Carretera convencional
- Autovía
- Carretera multicarril
- Autopista libre
- Autopista de peaje
- Autopista libre de peaje para movimientos internos
- Autopista con peaje bonificado

## Filtering logic adopted (v1)
Segments are kept if either:
1. `clase` belongs to one of the motorway / high-capacity categories:
   - Autovía
   - Autopista libre
   - Autopista de peaje
   - Autopista libre de peaje para movimientos internos
   - Autopista con peaje bonificado

OR

2. `carretera` starts with `N-`, used as a proxy for **carreteras nacionales**

## Justification
The datathon brief explicitly focuses the solution on interurban roads, specifically autopistas, autovías, and carreteras nacionales.

`clase` captures motorway-type roads directly, but carreteras nacionales do not appear as a dedicated class. Instead, they are identified through the `carretera` field using the `N-*` naming pattern.

This avoids two methodological errors:
- keeping all `Carretera convencional` segments, which would over-expand the network with many non-national roads
- excluding all `Carretera multicarril` segments, which would wrongly remove valid national corridors such as `N-1`, `N-2`, `N-120`, etc.

## Important caveat
`Autopista libre de peaje para movimientos internos` is kept in this version to preserve corridor continuity in the backbone, but these segments may later be excluded or downweighted during candidate generation if they behave like connectors or internal movements rather than meaningful charging locations.

## Interpretation
This layer is the **mobility backbone**, not the final siting layer.

It defines:
- where existing charging stations will be matched
- where corridor coverage will be measured
- where infrastructure gaps will be assessed

It does **not yet** define the final locations where new stations should be proposed.

## Outputs created
- `mobility_backbone.gpkg`
- `mobility_backbone.csv`

## Pending next step
Parse the official charging-point baseline and match it to this mobility backbone to identify existing interurban coverage and corridor gaps.

In [8]:
import zipfile
import os

zip_path = "road_routes_tramos_ministry.zip"
extract_dir = "road_routes_tramos_ministry"

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_dir)

print(os.listdir(extract_dir))

['250311 catalogoRCE2024.shp', '250311 catalogoRCE2024.cpg', '250311 catalogoRCE2024.shx', '250311 catalogoRCE2024.prj', '250311 catalogoRCE2024.dbf']


In [19]:
import geopandas as gpd

gdf = gpd.read_file("road_routes_tramos_ministry/250311 catalogoRCE2024.shp")
print(gdf.columns.tolist())
gdf.head(10)
gdf['clase'].value_counts()

['id_catalog', 'provincia', 'carretera', 'pk_inicio', 'pk_fin', 'longitud', 'pkinicio_t', 'pkfin_t', 'inicio', 'fin', 'clase', 'geometry']


,count
clase,
Carretera convencional,4345
Autovía,2048
Carretera multicarril,285
Autopista libre,180
Autopista de peaje,159
Autopista libre de peaje para movimientos internos,56
Autopista con peaje bonificado,5


In [20]:
import pandas as pd

# unique examples by class
gdf.groupby('clase')['carretera'].unique().apply(lambda x: list(x[:20]))

,carretera
clase,
Autopista con peaje bonificado,[AP-9]
Autopista de peaje,"[B-23, AP-7, AP-46, AP-41, R-5, R-4, AP-36, AP..."
Autopista libre,"[AP-7, AP-2, B-23, A-43, AP-4, AP-4A, AP-36, A..."
Autopista libre de peaje para movimientos internos,"[A-7, A-40, AP-7, AP-9, AP-9V, AP-51, AP-6, M-..."
Autovía,"[A-62, A-4, A-6, LL-12, PO-11, A-56, A-60, A-6..."
Carretera convencional,"[N-260, N-432, N-554, N-601, N-622, N-122, N-3..."
Carretera multicarril,"[N-550, H-30, N-6, AC-11, BA-20, N-1, N-2, N-4..."


In [21]:
conv = gdf[gdf['clase'] == 'Carretera convencional'].copy()
conv['is_nacional'] = conv['carretera'].astype(str).str.startswith('N-')

conv['is_nacional'].value_counts(dropna=False)
conv[['carretera', 'provincia']].drop_duplicates().sort_values('carretera').head(50)

,carretera,provincia
514,A-11R,Burgos
6109,A-14-R2,Lleida
4260,A-22-R1,Lleida
4860,A-22-R2,Lleida
570,A-26,Girona
515,A-4R1,Cádiz
271,A-66R,León
624,AC-10,A Coruña
1530,AC-14AL,A Coruña
2050,AC-14T,A Coruña


In [22]:
multi = gdf[gdf['clase'] == 'Carretera multicarril'].copy()
multi[['carretera', 'provincia']].drop_duplicates().sort_values('carretera').head(50)

,carretera,provincia
5231,A-1A,Madrid
5612,A-2,Girona
324,A-4R,Córdoba
2224,A-55,Pontevedra
519,A-68,Zaragoza
954,A-7,Málaga
775,A-7,Cádiz
278,A-73R,Burgos
1774,A-77A,Alicante/Alacant
1872,A-79,Alicante/Alacant


In [23]:
autopista_autovia_classes = [
    'Autovía',
    'Autopista libre',
    'Autopista de peaje',
    'Autopista libre de peaje para movimientos internos',
    'Autopista con peaje bonificado'
]

gdf['is_nacional'] = gdf['carretera'].astype(str).str.startswith('N-')

mobility_gdf = gdf[
    gdf['clase'].isin(autopista_autovia_classes) | gdf['is_nacional']
].copy()

In [24]:
print("Original segments:", len(gdf))
print("Filtered segments:", len(mobility_gdf))
print("\nClasses kept:")
print(mobility_gdf['clase'].value_counts(dropna=False))

print("\nSample roads kept:")
print(
    mobility_gdf[['carretera', 'clase', 'provincia']]
    .drop_duplicates()
    .sort_values(['carretera', 'provincia'])
    .head(50)
)

Original segments: 7078
Filtered segments: 6896

Classes kept:
clase
Carretera convencional                                4304
Autovía                                               2048
Autopista libre                                        180
Autopista de peaje                                     159
Carretera multicarril                                  144
Autopista libre de peaje para movimientos internos      56
Autopista con peaje bonificado                           5
Name: count, dtype: int64

Sample roads kept:
     carretera    clase           provincia
464        A-1  Autovía              Burgos
1003       A-1  Autovía              Madrid
513        A-1  Autovía             Segovia
272       A-11  Autovía              Burgos
376       A-11  Autovía               Soria
5417      A-11  Autovía          Valladolid
5007      A-11  Autovía              Zamora
238       A-12  Autovía            La Rioja
1477      A-13  Autovía            La Rioja
5963      A-14  Autovía         

In [25]:
mobility_gdf.to_file("mobility_backbone.gpkg", driver="GPKG")
mobility_gdf.drop(columns='geometry').to_csv("mobility_backbone.csv", index=False)